# Medallion Data Validation

This notebook validates that the Bronze, Silver, and Gold layers conform to the pipeline requirements and that extract/processing was successful.

## Setup

In [1]:
import json
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import pyarrow.parquet as pq

# Support running from repo root (data/) or from notebooks/ (../data)
DATA_ROOT = Path("data") if (Path("data") / "bronze").exists() else Path("../data")
DATA_ROOT = DATA_ROOT.resolve()
BRONZE_BASE = DATA_ROOT / "bronze" / "breweries"
SILVER_PATH = DATA_ROOT / "silver" / "breweries"
GOLD_PATH = DATA_ROOT / "gold" / "breweries_by_type_location"

REQUIRED_FIELDS = {"id", "name", "brewery_type", "city", "state_province", "country", "postal_code", "state"}
PARTITION_KEYS = {"country", "state_province"}
SILVER_COLUMNS = {
    "id", "name", "brewery_type", "address_1", "address_2", "address_3",
    "city", "state_province", "postal_code", "country", "longitude", "latitude",
    "phone", "website_url", "ingested_at", "source_file"
}
GOLD_SCHEMA = {"brewery_type", "country", "state_province", "brewery_count", "aggregated_at"}

def get_latest_bronze_run() -> Path:
    """Return path to the most recent Bronze run directory."""
    if not BRONZE_BASE.exists():
        raise FileNotFoundError(f"Bronze base not found: {BRONZE_BASE}")
    runs = sorted(BRONZE_BASE.glob("run_id=*"))
    if not runs:
        raise FileNotFoundError(f"No run_id directories under {BRONZE_BASE}")
    return runs[-1]

def get_delta_current_files(table_path: Path) -> set[str]:
    """Return set of current data file paths (relative to table_path) from Delta _delta_log."""
    log_dir = table_path / "_delta_log"
    if not log_dir.exists():
        return set()
    json_files = sorted(log_dir.glob("*.json"))
    current = set()
    for jf in json_files:
        for line in jf.read_text().strip().split("\n"):
            if not line.strip():
                continue
            obj = json.loads(line)
            if "add" in obj:
                current.add(unquote(obj["add"]["path"]))
            if "remove" in obj:
                current.discard(unquote(obj["remove"]["path"]))
    return current

---
## 1. Bronze Layer

**Requirements (from PLAN):** JSONL, one file per run; required fields: `id`, `name`, `brewery_type`, `city`, `state_province`, `country`, `postal_code`, `state`; partition keys non-null.

In [2]:
run_dir = get_latest_bronze_run()
bronze_file = run_dir / "breweries.jsonl"
print(f"Bronze run: {run_dir.name}")
print(f"File: {bronze_file}")
print(f"Exists: {bronze_file.exists()}")

Bronze run: run_id=manual__2026-03-07T13:57:01+00:00
File: /Users/adelaideadurens/Documents/Github-Adelaide-Adurens/breweries-api-data-pipeline/data/bronze/breweries/run_id=manual__2026-03-07T13:57:01+00:00/breweries.jsonl
Exists: True


In [3]:
bronze_records = []
for line in bronze_file.read_text(encoding="utf-8").strip().split("\n"):
    if line.strip():
        bronze_records.append(json.loads(line))

bronze_count = len(bronze_records)
print(f"Total records: {bronze_count}")

Total records: 9251


In [4]:
sample = bronze_records[0]
print("Sample record keys:", sorted(sample.keys()))
missing = REQUIRED_FIELDS - set(sample.keys())
null_or_empty = [k for k in REQUIRED_FIELDS if k in sample and (sample[k] is None or (isinstance(sample[k], str) and not str(sample[k]).strip()))]
print(f"Required fields present: {REQUIRED_FIELDS.issubset(sample.keys())}")
print(f"Missing required: {missing or 'None'}")
print(f"Null/empty required: {null_or_empty or 'None'}")

Sample record keys: ['address_1', 'address_2', 'address_3', 'brewery_type', 'city', 'country', 'id', 'latitude', 'longitude', 'name', 'phone', 'postal_code', 'state', 'state_province', 'street', 'website_url']
Required fields present: True
Missing required: None
Null/empty required: None


In [5]:
all_have_required = all(REQUIRED_FIELDS.issubset(r.keys()) for r in bronze_records)
partition_ok = all(
    r.get(k) is not None and str(r.get(k, "")).strip() != ""
    for r in bronze_records for k in PARTITION_KEYS
)
print(f"All records have required fields: {all_have_required}")
print(f"Partition keys non-null/non-empty: {partition_ok}")
print("\nBronze validation: PASS" if (all_have_required and partition_ok) else "\nBronze validation: CHECK FAILED")

All records have required fields: True
Partition keys non-null/non-empty: True

Bronze validation: PASS


---
## 2. Silver Layer

**Requirements:** Delta Lake; partitioned by `country`, `state_province`; columns normalized (no deprecated state/street); deduplicated by `id`.

In [6]:
# Read Silver using Delta log so we only read current-version files (no orphan parquet from overwrites)
current_files = get_delta_current_files(SILVER_PATH)
if not current_files:
    raise FileNotFoundError(f"No current data files in Delta table at {SILVER_PATH}")
dfs = [pq.read_table(SILVER_PATH / p).to_pandas() for p in current_files]
silver_df = pd.concat(dfs, ignore_index=True)
print(f"Silver table path: {SILVER_PATH}")
print(f"Total rows: {len(silver_df)}")
print(f"Columns: {list(silver_df.columns)}")

Silver table path: /Users/adelaideadurens/Documents/Github-Adelaide-Adurens/breweries-api-data-pipeline/data/silver/breweries
Total rows: 9251
Columns: ['id', 'name', 'brewery_type', 'address_1', 'address_2', 'address_3', 'city', 'postal_code', 'longitude', 'latitude', 'phone', 'website_url', 'ingested_at', 'source_file', 'country', 'state_province']


In [7]:
partition_cols_present = PARTITION_KEYS.issubset(silver_df.columns)
expected_columns = SILVER_COLUMNS.issubset(silver_df.columns)
no_deprecated = "state" not in silver_df.columns and "street" not in silver_df.columns
dedup_ok = silver_df["id"].nunique() == len(silver_df)
print(f"Partition columns (country, state_province) present: {partition_cols_present}")
print(f"Expected Silver columns present: {expected_columns}")
print(f"Deprecated (state, street) dropped: {no_deprecated}")
print(f"Deduplicated by id (no duplicate ids): {dedup_ok}")
print("\nSilver validation: PASS" if all([partition_cols_present, expected_columns, no_deprecated, dedup_ok]) else "\nSilver validation: CHECK FAILED")

Partition columns (country, state_province) present: True
Expected Silver columns present: True
Deprecated (state, street) dropped: True
Deduplicated by id (no duplicate ids): True

Silver validation: PASS


In [8]:
print("Partition value counts (top 10 by count):")
silver_df.groupby(["country", "state_province"]).size().sort_values(ascending=False).head(10)

Partition value counts (top 10 by count):


country        state_province
United States  California        919
               Washington        498
               Colorado          449
               New York          419
               Michigan          375
               Texas             352
               Pennsylvania      345
               Florida           312
               North Carolina    311
               Ohio              303
dtype: int64

---
## 3. Gold Layer

**Requirements:** Parquet; schema `brewery_type`, `country`, `state_province`, `brewery_count`, `aggregated_at`; aggregated count per type and location.

In [9]:
gold_file = GOLD_PATH / "breweries_by_type_location.parquet"
gold_df = pd.read_parquet(gold_file)
print(f"Gold path: {gold_file}")
print(f"Rows: {len(gold_df)}")
print(f"Columns: {list(gold_df.columns)}")

Gold path: /Users/adelaideadurens/Documents/Github-Adelaide-Adurens/breweries-api-data-pipeline/data/gold/breweries_by_type_location/breweries_by_type_location.parquet
Rows: 455
Columns: ['brewery_type', 'country', 'state_province', 'brewery_count', 'aggregated_at']


In [10]:
schema_ok = set(gold_df.columns) == GOLD_SCHEMA
total_gold_count = gold_df["brewery_count"].sum()
matches_silver = total_gold_count == len(silver_df)
print(f"Schema equals {GOLD_SCHEMA}: {schema_ok}")
print(f"Sum of brewery_count in Gold: {total_gold_count}")
print(f"Silver row count: {len(silver_df)}")
print(f"Gold brewery_count sum matches Silver rows: {matches_silver}")
print("\nGold validation: PASS" if (schema_ok and matches_silver) else "\nGold validation: CHECK FAILED")

Schema equals {'aggregated_at', 'country', 'brewery_count', 'brewery_type', 'state_province'}: True
Sum of brewery_count in Gold: 9251
Silver row count: 9251
Gold brewery_count sum matches Silver rows: True

Gold validation: PASS


In [11]:
print("Gold sample (top 15 by brewery_count):")
gold_df.nlargest(15, "brewery_count")

Gold sample (top 15 by brewery_count):


,brewery_type,country,state_province,brewery_count,aggregated_at
268,micro,United States,California,466,2026-03-08 14:37:16.125595+00:00
50,brewpub,United States,California,254,2026-03-08 14:37:16.125595+00:00
312,micro,United States,Washington,245,2026-03-08 14:37:16.125595+00:00
269,micro,United States,Colorado,228,2026-03-08 14:37:16.125595+00:00
297,micro,United States,New York,226,2026-03-08 14:37:16.125595+00:00
68,brewpub,United States,Michigan,199,2026-03-08 14:37:16.125595+00:00
308,micro,United States,Texas,196,2026-03-08 14:37:16.125595+00:00
273,micro,United States,Florida,190,2026-03-08 14:37:16.125595+00:00
298,micro,United States,North Carolina,185,2026-03-08 14:37:16.125595+00:00
300,micro,United States,Ohio,163,2026-03-08 14:37:16.125595+00:00


---
## Summary

In [12]:
summary = {
    "Bronze": {"records": bronze_count, "required_fields_ok": all_have_required, "partition_ok": partition_ok},
    "Silver": {"rows": len(silver_df), "partition_ok": partition_cols_present, "dedup_ok": dedup_ok},
    "Gold": {"rows": len(gold_df), "schema_ok": schema_ok, "count_matches_silver": matches_silver},
}
pd.DataFrame.from_dict(summary, orient="index")

,records,required_fields_ok,partition_ok,rows,dedup_ok,schema_ok,count_matches_silver
Bronze,9251.0,True,True,NaN,NaN,NaN,NaN
Silver,NaN,NaN,True,9251.0,True,NaN,NaN
Gold,NaN,NaN,NaN,455.0,NaN,True,True
